In [5]:
import pandas as pd
from sqlalchemy import Date, cast, func, select

from models.app_models import Commande, LigneCommande, Magasin
from models.dw_models import DimDate, DimMagasin, DimProduit, FaitVente
from utils.api import get_all_meteo
from utils.connections import app_session_factory, dw_session_factory


In [6]:
with dw_session_factory() as dw_session:
    df_dates = pd.read_sql(select(DimDate), con=dw_session.connection())
    df_produits = pd.read_sql(select(DimProduit), con=dw_session.connection())
    df_magasins = pd.read_sql(select(DimMagasin), con=dw_session.connection())

with app_session_factory() as app_session:
    date_bounds_query = select(
        func.min(cast(Commande.date_commande, Date)).label("min_date"),
        func.max(cast(Commande.date_commande, Date)).label("max_date"),
    )
    res = app_session.execute(date_bounds_query).one()
    global_start = res.min_date
    global_end = res.max_date

meteo_records = await get_all_meteo(df_magasins, global_start, global_end)
df_meteo = pd.DataFrame(meteo_records) if meteo_records else pd.DataFrame()

display(df_magasins)
display(df_dates)
display(df_produits)
display(df_meteo)

,magasin_key,magasin_id,nom,adresse,ville,est_cotier,date_ouverture,code_postal,region,latitude,longitude
0,65,1,Givre & Co - Grand-Place,Rue du Marché aux Herbes 25,Bruxelles,False,2019-03-15,1000,Brussel-Hoofdstad - Bruxelles-Capitale,50.848065,4.352270
1,66,2,Givre & Co - Carré,Place du Marché 15,Liege,False,2020-04-01,4600,Liège,50.733828,5.697437
2,67,3,Givre & Co - Namur,Rue de Fer 30,Namur,False,2021-05-01,5000,Namur,50.465762,4.865540
3,68,4,Givre & Co - Gent,Veldstraat 50,Gent,False,2022-03-20,9890,Oost-Vlaanderen,50.919359,3.645297
4,69,5,Givre & Co - Digue,Albert I-Promenade 40,Oostende,True,2018-04-01,8400,West-Vlaanderen,51.234280,2.915651
5,70,6,Givre & Co - Zoute,Lippenslaan 100,Knokke heist,True,2023-04-01,8300,West-Vlaanderen,51.344796,3.288459
6,71,7,Givre & Co - Charleroi,Rue de la Montagne 20,Charleroi,False,2025-04-12,6200,Hainaut,50.404723,4.522594


,date_key,date_complete,jour,mois,nom_mois,trimestre,annee,semaine_iso,jour_semaine,nom_jour,est_weekend,est_ferie,saison
0,20240101,2024-01-01,1,1,janvier,1,2024,1,1,lundi,False,True,hiver
1,20240102,2024-01-02,2,1,janvier,1,2024,1,2,mardi,False,False,hiver
2,20240103,2024-01-03,3,1,janvier,1,2024,1,3,mercredi,False,False,hiver
3,20240104,2024-01-04,4,1,janvier,1,2024,1,4,jeudi,False,False,hiver
4,20240105,2024-01-05,5,1,janvier,1,2024,1,5,vendredi,False,False,hiver
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1091,20261227,2026-12-27,27,12,décembre,4,2026,52,7,dimanche,True,False,hiver
1092,20261228,2026-12-28,28,12,décembre,4,2026,53,1,lundi,False,False,hiver
1093,20261229,2026-12-29,29,12,décembre,4,2026,53,2,mardi,False,False,hiver
1094,20261230,2026-12-30,30,12,décembre,4,2026,53,3,mercredi,False,False,hiver


,produit_key,produit_id,code,nom,categorie,type_produit,prix_catalogue,est_actif,allergenes,est_vegan
0,78,1,GL-VAN,Cornet vanille,Glaces,Froid,2.8,True,"Lactose, Gluten, Fruits à coque",False
1,79,2,GL-CHO,Cornet chocolat,Glaces,Froid,2.8,True,"Lactose, Gluten, Fruits à coque",False
2,80,3,GL-SPE,Cornet spéculoos,Glaces,Froid,3.0,True,"Gluten, Lait, Fruits à coque",False
3,81,4,GL-PIS,Cornet pistache,Glaces,Froid,3.2,True,"Lait, Fruits à coques",False
4,82,5,GL-CUP,Coupe 3 boules,Glaces,Froid,6.5,True,"Lait, Fruits à coque",False
5,83,6,SO-FRA,Sorbet fraise,Sorbets,Froid,2.8,True,Peut contenir des traces de fruits à coque,True
6,84,7,SO-CIT,Sorbet citron,Sorbets,Froid,2.8,True,Aucun allergène courant n'est généralement pré...,True
7,85,8,SO-MAN,Sorbet mangue,Sorbets,Froid,3.0,True,Aucun,True
8,86,9,GA-LIE,Gaufre de liège,Gaufres & crêpes,Chaud,3.5,True,"Blé, Lactose, Oeufs",False
9,87,10,GA-BRU,Gaufre de bruxelles,Gaufres & crêpes,Chaud,4.5,True,"Gluten, Lait, Oeufs",False


,magasin_key,date_complete,temp_max,temp_min,precipitation_mm
0,65,2025-01-02,5.5,1.1,10.1
1,65,2025-01-03,2.7,-1.1,0.0
2,65,2025-01-04,2.3,-0.7,0.0
3,65,2025-01-05,11.6,0.8,26.0
4,65,2025-01-06,11.6,3.3,4.1
...,...,...,...,...,...
2543,71,2025-12-27,4.0,-1.9,0.0
2544,71,2025-12-28,4.4,-1.1,0.0
2545,71,2025-12-29,2.3,-3.4,0.0
2546,71,2025-12-30,3.6,-1.0,0.0


In [7]:
query = (
    select(
        LigneCommande.produit_id,
        Commande.magasin_id,
        cast(Commande.date_commande, Date).label("date_commande"),
        func.sum(LigneCommande.quantite).label("quantite"),
        func.count(Commande.commande_id.distinct()).label("nb_commandes"),
        func.sum(LigneCommande.quantite * LigneCommande.prix_unitaire).label(
            "montant_brut"
        ),
        func.sum(
            LigneCommande.quantite
            * LigneCommande.prix_unitaire
            * LigneCommande.remise_pct
            / 100
        ).label("montant_remise"),
    )
    .join(Commande)
    .join(Magasin)
    .where(Magasin.actif)
    .group_by(
        LigneCommande.produit_id,
        Commande.magasin_id,
        cast(Commande.date_commande, Date),
    )
)

with app_session_factory() as app_session:
    df = pd.read_sql(query, app_session.connection())

df

,produit_id,magasin_id,date_commande,quantite,nb_commandes,montant_brut,montant_remise
0,1,1,2025-01-14,3,1,7.8,0.78
1,1,1,2025-01-24,2,1,5.2,0.00
2,1,1,2025-02-07,1,1,2.6,0.00
3,1,1,2025-02-23,4,2,10.4,0.00
4,1,1,2025-03-08,1,1,2.6,0.00
...,...,...,...,...,...,...,...
25614,19,1,2025-02-12,3,1,8.4,0.00
25615,19,2,2025-02-09,1,1,2.8,0.00
25616,19,3,2025-01-10,1,1,2.8,0.00
25617,19,4,2025-02-01,3,1,8.4,0.00


In [ ]:
df["montant_net"] = df["montant_brut"] - df["montant_remise"]

df = df.merge(
    df_dates[['date_key', 'date_complete']], 
    left_on='date_commande', 
    right_on='date_complete', 
    how='left'
)

df = df.merge(
    df_produits[['produit_key', 'produit_id']], 
    on='produit_id', 
    how='left'
)

df = df.merge(
    df_magasins[['magasin_key', 'magasin_id']], 
    on='magasin_id', 
    how='left'
)

df = df.merge(
    df_meteo, on=["magasin_key", "date_complete"], how="left"
)

df

In [9]:
with dw_session_factory() as dw_session:
    df.drop(columns=["produit_id", "magasin_id", "date_commande", "date_complete"]).to_sql(
        FaitVente.__tablename__, 
        if_exists='append', 
        index=False, 
        con=dw_session.connection(),
    )
    dw_session.commit()